# Teste 5 da PoC: FedAvg em claro × SMPC × CKKS (mesmo Autoencoder)

**Objetivo:** medir o custo e o impacto na qualidade de cada mecanismo de agregação segura,
comparando **FedAvg em claro × SMPC** e **FedAvg em claro × CKKS** com o **mesmo modelo Autoencoder** (arquitetura do Teste 2).

**Controle de variáveis:** as três execuções usam o mesmo dataset, o mesmo split 60/20/20 estratificado,
o mesmo scaler (ajustado só no treino), a mesma partição entre clientes, os mesmos pesos iniciais
e as mesmas sementes de treino local. **A única diferença entre elas é o mecanismo de agregação.**

| Mecanismo | O que o servidor vê | Como a média é feita |
|---|---|---|
| FedAvg em claro | os pesos de cada cliente | soma ponderada em float64 (referência exata) |
| SMPC | apenas somas parciais mascaradas | secret sharing aditivo em campo finito (primo de Mersenne 2^61−1, ponto fixo) |
| CKKS | apenas cifrados (contexto **sem** chave secreta) | soma homomórfica no TenSEAL; só os clientes decifram |

Em todos os casos o FedAvg é **ponderado** (peso de cada cliente = n_k / N), aplicado pelo próprio cliente antes de proteger seus pesos.

**Métricas:** ROC-AUC, PR-AUC, Precisão, Recall, F1 e Acurácia no teste (limiar escolhido na validação);
erro numérico da agregação em relação ao FedAvg exato em cada rodada; concordância das predições com o modelo em claro;
tempo de proteção, agregação e revelação; e bytes enviados por cliente por rodada.

In [ ]:
!pip install -q tenseal

In [ ]:
# -*- coding: utf-8 -*-
"""
Teste 5 da PoC: Comparação de Agregação Segura em Aprendizado Federado
FedAvg em claro × SMPC  e  FedAvg em claro × CKKS (TenSEAL)
Modelo: Autoencoder Tabular (PyTorch), o mesmo do Teste 2
"""

import os
import copy
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, precision_recall_curve,
                             roc_curve)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils import parameters_to_vector, vector_to_parameters

try:
    import tenseal as ts
except ImportError:
    raise ImportError("TenSEAL não está instalado. Execute: pip install tenseal")

# =====================================================================
# --- 0. CONFIGURAÇÃO DO EXPERIMENTO ---
# =====================================================================
SEED = 42
NUM_CLIENTES = 3
NUM_RODADAS = 5          # Rodadas de comunicação do FL
EPOCAS_LOCAIS = 2        # Épocas de treino local por rodada
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-5

# Bases a executar (a que não for encontrada é pulada com aviso)
DATASETS = ['creditcard', 'paysim']
PAYSIM_AMOSTRA = 1_000_000   # Amostra estratificada da PaySim (None = base completa)

CAMINHOS = {
    'creditcard': [
        '/content/drive/MyDrive/Mestrado Cesar/Base de Dados/creditcard/creditcard.csv',
        'base_de_dados/creditcard.zip',        # Executando de dentro de PoC/
        'PoC/base_de_dados/creditcard.zip',    # Executando da raiz do repositório
    ],
    'paysim': [
        '/content/drive/MyDrive/Mestrado Cesar/Base de Dados/Kaggle/paysim.csv',
    ],
}

# SMPC: secret sharing aditivo em Z_p
SMPC_MODULO = 2**61 - 1        # Primo de Mersenne: cabe em int64 mesmo somando 3 shares
SMPC_BITS_PRECISAO = 24        # Ponto fixo: resolução de 2^-24 (mesmo valor do Teste 4)
SMPC_ESCALA = 2 ** SMPC_BITS_PRECISAO

# CKKS (TenSEAL): mesmos parâmetros do Teste 3
CKKS_POLY_MOD = 8192
CKKS_COEFF_MOD = [60, 40, 40, 60]
CKKS_ESCALA_BITS = 40


def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo PyTorch: {device}")


def sincronizar():
    """Garante que o cronômetro só pare depois que a GPU terminar (CUDA é assíncrono)."""
    if device.type == 'cuda':
        torch.cuda.synchronize()


try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Fora do Google Colab: usando apenas caminhos locais.")

In [ ]:
# =====================================================================
# --- 1. CARREGAMENTO E PREPARAÇÃO DOS DADOS ---
# =====================================================================

def ler_base(nome):
    for caminho in CAMINHOS[nome]:
        if os.path.exists(caminho):
            print(f"Carregando '{caminho}'...")
            return pd.read_csv(caminho)
    return None


def preparar_creditcard(df):
    # Time é um contador monotônico: ruído para modelos baseados em distância/reconstrução
    df = df.drop(columns=['Time'], errors='ignore').dropna(subset=['Class'])
    df['amount_log'] = np.log1p(df['Amount'].clip(lower=0))
    df = df.drop(columns=['Amount'])
    return df, 'Class'


def preparar_paysim(df):
    # Fraudes na PaySim ocorrem apenas em TRANSFER e CASH_OUT (EDA do Teste 1)
    df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
    df = df.drop(columns=[c for c in ['nameOrig', 'nameDest', 'isFlaggedFraud', 'step'] if c in df.columns])
    df = df.dropna(subset=['isFraud'])
    if PAYSIM_AMOSTRA and len(df) > PAYSIM_AMOSTRA:
        print(f"Amostragem estratificada de {PAYSIM_AMOSTRA:,} linhas (preserva a taxa de fraude)...")
        df, _ = train_test_split(df, train_size=PAYSIM_AMOSTRA, stratify=df['isFraud'], random_state=SEED)
    df['amount_log'] = np.log1p(df['amount'].clip(lower=0))
    df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
    df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
    df = df.drop(columns=['amount'])
    df = pd.get_dummies(df, columns=['type'], drop_first=True, dtype=float)
    return df, 'isFraud'


PREPARADORES = {'creditcard': preparar_creditcard, 'paysim': preparar_paysim}


def carregar_dados(nome):
    df = ler_base(nome)
    if df is None:
        return None
    df, alvo = PREPARADORES[nome](df)
    for col in df.select_dtypes(include=np.number).columns:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    X = df.drop(columns=[alvo]).values.astype(np.float32)
    y = df[alvo].values.astype(np.int64)

    # Split 60/20/20 estratificado ANTES de escalar (sem data leakage)
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp)

    scaler = RobustScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    # Autoencoder treina só com transações normais; cada cliente recebe uma fatia (IID)
    X_train_normal = X_train[y_train == 0]
    clientes_X = np.array_split(X_train_normal, NUM_CLIENTES)
    n_total = sum(len(c) for c in clientes_X)
    pesos_fedavg = [len(c) / n_total for c in clientes_X]

    print(f"Treino: {X_train.shape} | Validação: {X_val.shape} | Teste: {X_test.shape} "
          f"| Fraudes no teste: {int(y_test.sum())}")
    print(f"Amostras normais por cliente: {[len(c) for c in clientes_X]}")
    return dict(nome=nome, X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test,
                clientes_X=clientes_X, pesos_fedavg=pesos_fedavg, input_dim=X_train.shape[1])

In [ ]:
# =====================================================================
# --- 2. MODELO: AUTOENCODER TABULAR (MESMA ARQUITETURA DO TESTE 2) ---
# =====================================================================

class TabularAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(TabularAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 24),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.1),
            nn.Linear(24, 12),
            nn.LeakyReLU(0.01),
            nn.Linear(12, 6)  # Espaço latente (gargalo)
        )
        self.decoder = nn.Sequential(
            nn.Linear(6, 12),
            nn.LeakyReLU(0.01),
            nn.Linear(12, 24),
            nn.LeakyReLU(0.01),
            nn.Linear(24, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def modelo_para_vetor(modelo):
    return parameters_to_vector(modelo.parameters()).detach().cpu().to(torch.float64)


def vetor_para_modelo(vetor, modelo):
    vector_to_parameters(vetor.to(dtype=torch.float32, device=device), modelo.parameters())


def treinar_local(estado_global, dados_X, input_dim, rodada, cliente_id):
    """Treino local idêntico para todos os mecanismos: mesma semente por (rodada, cliente)."""
    semente = SEED + 1000 * rodada + cliente_id
    torch.manual_seed(semente)  # Controla o Dropout
    modelo = TabularAutoencoder(input_dim).to(device)
    modelo.load_state_dict(estado_global)
    modelo.train()
    optimizer = optim.Adam(modelo.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.MSELoss()

    gerador = torch.Generator().manual_seed(semente)  # Controla o shuffle
    loader = DataLoader(TensorDataset(torch.from_numpy(dados_X)), batch_size=BATCH_SIZE,
                        shuffle=True, generator=gerador)
    for _ in range(EPOCAS_LOCAIS):
        for (lote,) in loader:
            lote = lote.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(modelo(lote), lote)
            loss.backward()
            optimizer.step()
    return modelo_para_vetor(modelo)


def erros_reconstrucao(modelo, X, tamanho_lote=65536):
    modelo.eval()
    saida = []
    with torch.no_grad():
        for i in range(0, len(X), tamanho_lote):
            lote = torch.from_numpy(X[i:i + tamanho_lote]).to(device)
            saida.append(torch.mean((lote - modelo(lote)) ** 2, dim=1).cpu().numpy())
    return np.concatenate(saida)

In [ ]:
# =====================================================================
# --- 3. MECANISMOS DE AGREGAÇÃO ---
# Todos recebem os mesmos vetores de pesos locais e os pesos do FedAvg,
# e devolvem (vetor agregado, custos da rodada).
# =====================================================================

class AgregadorClaro:
    """FedAvg em claro: o servidor recebe os pesos de cada cliente sem proteção e faz a média ponderada."""
    nome = 'FedAvg em claro'

    def __init__(self):
        self.tempo_setup = 0.0

    def agregar(self, vetores, pesos):
        t0 = time.perf_counter()
        pacotes = [v.to(torch.float32) for v in vetores]  # Pesos brutos do modelo, enviados em float32
        t_protecao = (time.perf_counter() - t0) / len(vetores)

        t0 = time.perf_counter()
        agregado = sum(p.to(torch.float64) * a for p, a in zip(pacotes, pesos))
        t_agregacao = time.perf_counter() - t0

        bytes_cliente = pacotes[0].numel() * pacotes[0].element_size()
        return agregado, dict(t_protecao=t_protecao, t_agregacao=t_agregacao, t_revelacao=0.0,
                              bytes_cliente=bytes_cliente)


class AgregadorSMPC:
    """
    Secret sharing aditivo em Z_p com 3 nós (os próprios clientes).
    1. Cada cliente codifica seus pesos em ponto fixo e os divide em N shares aleatórios
       cuja soma (mod p) é o valor original; o share j vai para o nó j.
    2. Cada nó soma os shares recebidos (mod p): vê apenas ruído uniforme.
    3. O servidor soma as N somas parciais e obtém só o agregado, nunca um peso individual.
    """
    nome = 'SMPC'

    def __init__(self):
        self.tempo_setup = 0.0
        # Gerador próprio: não consome o RNG global, então o treino local fica idêntico ao das outras execuções.
        # Numa implantação real os shares devem vir de um CSPRNG (ex.: secrets / os.urandom).
        self.gerador = torch.Generator().manual_seed(SEED + 777)

    @staticmethod
    def codificar(v):
        return torch.remainder(torch.round(v * SMPC_ESCALA).to(torch.int64), SMPC_MODULO)

    @staticmethod
    def decodificar(v_int):
        v_int = torch.where(v_int > SMPC_MODULO // 2, v_int - SMPC_MODULO, v_int)
        return v_int.to(torch.float64) / SMPC_ESCALA

    def gerar_shares(self, v_int, n):
        shares = [torch.randint(0, SMPC_MODULO, v_int.shape, dtype=torch.int64, generator=self.gerador)
                  for _ in range(n - 1)]
        ultimo = v_int.clone()
        for s in shares:
            ultimo = torch.remainder(ultimo - s, SMPC_MODULO)
        return shares + [ultimo]

    def agregar(self, vetores, pesos):
        n = len(vetores)
        caixa_entrada = [[] for _ in range(n)]  # caixa_entrada[j]: shares recebidos pelo nó j

        # Clientes: codificam e distribuem os shares
        t0 = time.perf_counter()
        for v, a in zip(vetores, pesos):
            for j, share in enumerate(self.gerar_shares(self.codificar(v * a), n)):
                caixa_entrada[j].append(share)
        t_protecao = (time.perf_counter() - t0) / n

        # Nós: soma parcial cega (mod p); Servidor: soma das parciais
        t0 = time.perf_counter()
        somas_parciais = []
        for j in range(n):
            soma = torch.zeros_like(caixa_entrada[j][0])
            for share in caixa_entrada[j]:
                soma = torch.remainder(soma + share, SMPC_MODULO)
            somas_parciais.append(soma)
        total = torch.zeros_like(somas_parciais[0])
        for parcial in somas_parciais:
            total = torch.remainder(total + parcial, SMPC_MODULO)
        t_agregacao = time.perf_counter() - t0

        # Revelação: volta de Z_p para float
        t0 = time.perf_counter()
        agregado = self.decodificar(total)
        t_revelacao = time.perf_counter() - t0

        # Cada cliente envia (n-1) shares aos outros nós + 1 soma parcial ao servidor
        bytes_vetor = total.numel() * total.element_size()
        return agregado, dict(t_protecao=t_protecao, t_agregacao=t_agregacao, t_revelacao=t_revelacao,
                              bytes_cliente=n * bytes_vetor)


class AgregadorCKKS:
    """
    Criptografia homomórfica CKKS (TenSEAL).
    Os clientes compartilham o contexto com chave secreta; o servidor recebe apenas o contexto
    público (sem chave secreta) e soma os cifrados sem conseguir decifrá-los.
    """
    nome = 'CKKS'

    def __init__(self):
        t0 = time.perf_counter()
        self.ctx_cliente = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=CKKS_POLY_MOD,
                                      coeff_mod_bit_sizes=CKKS_COEFF_MOD)
        self.ctx_cliente.global_scale = 2 ** CKKS_ESCALA_BITS
        self.ctx_servidor = ts.context_from(self.ctx_cliente.serialize(save_secret_key=False))
        assert not self.ctx_servidor.has_secret_key(), "O servidor não pode ter a chave secreta!"
        self.slots = CKKS_POLY_MOD // 2
        self.tempo_setup = time.perf_counter() - t0

    def agregar(self, vetores, pesos):
        n = len(vetores)

        # Clientes: cifram (em blocos de 'slots' valores) e serializam para envio
        t0 = time.perf_counter()
        pacotes = []
        for v, a in zip(vetores, pesos):
            valores = (v * a).tolist()
            blocos = [valores[i:i + self.slots] for i in range(0, len(valores), self.slots)]
            pacotes.append([ts.ckks_vector(self.ctx_cliente, b).serialize() for b in blocos])
        t_protecao = (time.perf_counter() - t0) / n
        bytes_cliente = sum(len(b) for b in pacotes[0])

        # Servidor: desserializa com o contexto PÚBLICO e soma homomorficamente
        t0 = time.perf_counter()
        soma = [ts.ckks_vector_from(self.ctx_servidor, b) for b in pacotes[0]]
        for pacote in pacotes[1:]:
            for i, b in enumerate(pacote):
                soma[i] = soma[i] + ts.ckks_vector_from(self.ctx_servidor, b)
        agregado_serializado = [c.serialize() for c in soma]
        t_agregacao = time.perf_counter() - t0

        # Clientes: decifram o agregado com a chave secreta
        t0 = time.perf_counter()
        valores = []
        for b in agregado_serializado:
            valores.extend(ts.ckks_vector_from(self.ctx_cliente, b).decrypt())
        t_revelacao = time.perf_counter() - t0

        return torch.tensor(valores, dtype=torch.float64), dict(
            t_protecao=t_protecao, t_agregacao=t_agregacao, t_revelacao=t_revelacao,
            bytes_cliente=bytes_cliente)

In [ ]:
# =====================================================================
# --- 4. EXECUÇÃO DO FL E AVALIAÇÃO ---
# =====================================================================

def executar_fl(agregador, dados, estado_inicial):
    input_dim = dados['input_dim']
    modelo_global = TabularAutoencoder(input_dim).to(device)
    modelo_global.load_state_dict(estado_inicial)
    historico = []

    for rodada in range(NUM_RODADAS):
        estado = copy.deepcopy(modelo_global.state_dict())
        t0 = time.perf_counter()
        vetores = [treinar_local(estado, dados['clientes_X'][k], input_dim, rodada, k)
                   for k in range(NUM_CLIENTES)]
        sincronizar()
        t_treino = time.perf_counter() - t0

        agregado, custos = agregador.agregar(vetores, dados['pesos_fedavg'])

        # Referência: FedAvg exato (float64) dos MESMOS pesos locais
        referencia = sum(v * a for v, a in zip(vetores, dados['pesos_fedavg']))
        diff = (agregado - referencia).abs()

        vetor_para_modelo(agregado, modelo_global)
        auc_val = roc_auc_score(dados['y_val'], erros_reconstrucao(modelo_global, dados['X_val']))

        historico.append(dict(rodada=rodada + 1, t_treino=t_treino, erro_max=diff.max().item(),
                              erro_rmse=diff.pow(2).mean().sqrt().item(), roc_auc_val=auc_val, **custos))
        print(f"  Rodada {rodada + 1}/{NUM_RODADAS} | ROC-AUC val: {auc_val:.4f} "
              f"| erro máx. agregação: {diff.max().item():.2e} "
              f"| agregação: {1000 * (custos['t_protecao'] + custos['t_agregacao'] + custos['t_revelacao']):.1f} ms")
    return modelo_global, pd.DataFrame(historico)


def escolher_limiar(y_val, erros_val):
    """Limiar que maximiza o F1 na VALIDAÇÃO (nunca no teste)."""
    precisions, recalls, thresholds = precision_recall_curve(y_val, erros_val)
    f1 = np.divide(2 * precisions * recalls, precisions + recalls,
                   out=np.zeros_like(precisions), where=(precisions + recalls) != 0)
    idx = np.argmax(f1[:-1])  # O último ponto da curva não tem limiar associado
    return thresholds[idx]


def avaliar(modelo, dados):
    limiar = escolher_limiar(dados['y_val'], erros_reconstrucao(modelo, dados['X_val']))

    sincronizar()
    t0 = time.perf_counter()
    scores = erros_reconstrucao(modelo, dados['X_test'])
    sincronizar()
    t_inferencia = time.perf_counter() - t0

    y_test = dados['y_test']
    y_pred = (scores >= limiar).astype(int)
    return dict(
        limiar=limiar, scores=scores, y_pred=y_pred,
        **{'ROC-AUC': roc_auc_score(y_test, scores),
           'PR-AUC': average_precision_score(y_test, scores),
           'Precisão': precision_score(y_test, y_pred, zero_division=0),
           'Recall': recall_score(y_test, y_pred, zero_division=0),
           'F1': f1_score(y_test, y_pred, zero_division=0),
           'Acurácia': accuracy_score(y_test, y_pred),
           'TPS inferência': len(y_test) / t_inferencia})

In [ ]:
# =====================================================================
# --- 5. EXPERIMENTO: CLARO × SMPC E CLARO × CKKS ---
# =====================================================================

MECANISMOS = [AgregadorClaro, AgregadorSMPC, AgregadorCKKS]
METRICAS = ['ROC-AUC', 'PR-AUC', 'Precisão', 'Recall', 'F1', 'Acurácia']
CORES = {'FedAvg em claro': '#1f77b4', 'SMPC': '#ff7f0e', 'CKKS': '#2ca02c'}
resultados_gerais = {}

for nome_base in DATASETS:
    print("\n" + "=" * 78)
    print(f" BASE: {nome_base.upper()}")
    print("=" * 78)
    dados = carregar_dados(nome_base)
    if dados is None:
        print(f"Aviso: base '{nome_base}' não encontrada em {CAMINHOS[nome_base]}. Pulando.")
        continue

    # Pesos iniciais idênticos para os três mecanismos
    seed_everything(SEED)
    estado_inicial = copy.deepcopy(TabularAutoencoder(dados['input_dim']).to(device).state_dict())
    n_params = sum(t.numel() for t in estado_inicial.values())
    print(f"Autoencoder: {dados['input_dim']} features de entrada, {n_params:,} parâmetros treináveis.")

    execucoes = {}
    for Mecanismo in MECANISMOS:
        agregador = Mecanismo()
        print(f"\n--- {agregador.nome} (setup: {agregador.tempo_setup:.2f}s) ---")
        modelo, historico = executar_fl(agregador, dados, estado_inicial)
        execucoes[agregador.nome] = dict(modelo=modelo, historico=historico, vetor=modelo_para_vetor(modelo),
                                         avaliacao=avaliar(modelo, dados), tempo_setup=agregador.tempo_setup)

    # ----- Tabela comparativa -----
    claro = execucoes['FedAvg em claro']
    linhas = []
    for nome_mec, ex in execucoes.items():
        av, hist = ex['avaliacao'], ex['historico']
        overhead_ms = 1000 * (hist['t_protecao'] + hist['t_agregacao'] + hist['t_revelacao']).mean()
        linha = {'Mecanismo': nome_mec, **{m: av[m] for m in METRICAS},
                 'Erro máx. agregação': hist['erro_max'].max(),
                 'Δ máx. pesos finais vs claro': (ex['vetor'] - claro['vetor']).abs().max().item(),
                 'Concordância pred. vs claro (%)': 100 * np.mean(av['y_pred'] == claro['avaliacao']['y_pred']),
                 'Correlação scores vs claro': np.corrcoef(av['scores'], claro['avaliacao']['scores'])[0, 1],
                 'Overhead agregação (ms/rodada)': overhead_ms,
                 'Upload por cliente/rodada (KB)': hist['bytes_cliente'].iloc[0] / 1024,
                 'Setup (s)': ex['tempo_setup'],
                 'Treino local (s/rodada)': hist['t_treino'].mean()}
        linhas.append(linha)
    tabela = pd.DataFrame(linhas).set_index('Mecanismo')
    resultados_gerais[nome_base] = dict(tabela=tabela, execucoes=execucoes)

    pd.set_option('display.width', 200)
    print("\n>> MÉTRICAS DE QUALIDADE NO TESTE (limiar escolhido na validação):")
    print(tabela[METRICAS].map(lambda x: f"{x:.4f}").to_string())
    print("\n>> FIDELIDADE E CUSTO DA AGREGAÇÃO SEGURA:")
    print(tabela.drop(columns=METRICAS).to_string(float_format=lambda x: f"{x:.4g}"))

    print("\n>> COMPARAÇÕES PAREADAS (Δ = mecanismo seguro − claro):")
    for seguro in ['SMPC', 'CKKS']:
        t = tabela.loc[seguro]
        deltas = " | ".join(f"Δ{m}: {t[m] - tabela.loc['FedAvg em claro', m]:+.2e}" for m in ['ROC-AUC', 'PR-AUC', 'F1'])
        fator = t['Overhead agregação (ms/rodada)'] / max(tabela.loc['FedAvg em claro', 'Overhead agregação (ms/rodada)'], 1e-9)
        banda = t['Upload por cliente/rodada (KB)'] / tabela.loc['FedAvg em claro', 'Upload por cliente/rodada (KB)']
        print(f"  Claro × {seguro}: {deltas} | concordância {t['Concordância pred. vs claro (%)']:.3f}% "
              f"| overhead {fator:,.0f}x | banda {banda:.1f}x")

    # ----- Visualizações -----
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    fig.suptitle(f"Teste 5: FedAvg em claro × SMPC × CKKS ({nome_base})", fontsize=14, fontweight='bold')

    # 1. Métricas de qualidade
    ax = axes[0, 0]
    tabela[METRICAS].T.plot.bar(ax=ax, color=[CORES[m] for m in tabela.index], rot=0)
    ax.set_title('Métricas no teste')
    ax.set_ylim(0, 1.2)
    ax.legend(loc='upper center', ncol=3)

    # 2. Convergência
    ax = axes[0, 1]
    for nome_mec, ex in execucoes.items():
        estilo = '-' if nome_mec == 'FedAvg em claro' else '--'
        ax.plot(ex['historico']['rodada'], ex['historico']['roc_auc_val'], estilo, marker='o',
                color=CORES[nome_mec], label=nome_mec)
    ax.set_title('ROC-AUC na validação por rodada')
    ax.set_xlabel('Rodada')
    ax.legend()

    # 3. Curvas ROC no teste
    ax = axes[0, 2]
    for nome_mec, ex in execucoes.items():
        fpr, tpr, _ = roc_curve(dados['y_test'], ex['avaliacao']['scores'])
        estilo = '-' if nome_mec == 'FedAvg em claro' else '--'
        ax.plot(fpr, tpr, estilo, color=CORES[nome_mec], label=f"{nome_mec} (AUC={ex['avaliacao']['ROC-AUC']:.4f})")
    ax.plot([0, 1], [0, 1], color='gray', linestyle=':')
    ax.set_title('Curva ROC no teste')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(loc='lower right')

    # 4. Erro numérico da agregação
    ax = axes[1, 0]
    for nome_mec in ['SMPC', 'CKKS']:
        h = execucoes[nome_mec]['historico']
        ax.plot(h['rodada'], h['erro_max'], marker='o', color=CORES[nome_mec], label=f'{nome_mec} (máx.)')
        ax.plot(h['rodada'], h['erro_rmse'], marker='x', linestyle=':', color=CORES[nome_mec], label=f'{nome_mec} (RMSE)')
    ax.set_yscale('log')
    ax.set_title('Erro da agregação vs FedAvg exato')
    ax.set_xlabel('Rodada')
    ax.legend()

    # 5. Tempo de agregação por etapa
    ax = axes[1, 1]
    tempos = pd.DataFrame({nome_mec: 1000 * ex['historico'][['t_protecao', 't_agregacao', 't_revelacao']].mean()
                           for nome_mec, ex in execucoes.items()}).T
    tempos.columns = ['Proteção (por cliente)', 'Agregação', 'Revelação']
    tempos.plot.bar(stacked=True, ax=ax, rot=0, color=['#9467bd', '#8c564b', '#e377c2'])
    ax.set_yscale('symlog', linthresh=0.1)
    ax.set_title('Custo da agregação por rodada (ms)')

    # 6. Comunicação
    ax = axes[1, 2]
    kb = tabela['Upload por cliente/rodada (KB)']
    ax.bar(kb.index, kb.values, color=[CORES[m] for m in kb.index])
    for i, v in enumerate(kb.values):
        ax.text(i, v, f"{v:,.1f} KB", ha='center', va='bottom')
    ax.set_yscale('log')
    ax.set_title('Upload por cliente por rodada')

    plt.tight_layout()
    plt.show()

print("\nFim do Teste 5 (FedAvg em claro × SMPC × CKKS)!")

## Nova base: Healthcare Fraud Detection

As células abaixo foram **adicionadas** ao Teste 5, sem alterar as anteriores. Elas repetem o experimento da
seção 5 (FedAvg em claro × SMPC × CKKS com o mesmo Autoencoder) com a **mesma configuração** — modelo, hiperparâmetros, sementes, split, limiar e métricas — na base
`healthcare_fraud_detection.csv`, carregada direto do Google Drive
(`/content/drive/MyDrive/Mestrado Cesar/Base de Dados/Kaggle/`). O rótulo de fraude é a coluna `Is_Fraud`.

1. A primeira célula registra a base (caminho e engenharia de features) no pipeline do teste.
2. A segunda executa o experimento na base healthcare.
3. A terceira compara os resultados com as outras bases executadas neste notebook.

**Engenharia de features da base healthcare** (a mesma nos seis testes):
- removidos `Provider_ID` e `Claim_ID` (identificadores, como `nameOrig`/`nameDest` na PaySim) e `Claim_Submission_Date` (carimbo de tempo, como `Time` na Credit Card);
- `claim_amount_log` e `approved_amount_log` (log dos valores, como `amount_log`) e `approved_ratio` = valor aprovado ÷ valor solicitado (análoga ao `errorBalance` da PaySim);
- One-Hot das categóricas (`Patient_Gender`, `Diagnosis_Code`, `Procedure_Code`, `Insurance_Type`, `Provider_Specialty`, `Patient_State`, `Claim_Status`, `Visit_Type`), com ausentes na categoria `Desconhecido`;
- mediana para numéricas ausentes (`Prior_Visits_12m`).

In [ ]:
# =====================================================================
# --- NOVA BASE: Healthcare Fraud Detection (mesma configuração do teste) ---
# =====================================================================
CAMINHOS['healthcare'] = [
    '/content/drive/MyDrive/Mestrado Cesar/Base de Dados/Kaggle/healthcare_fraud_detection.csv',
    'base_de_dados/healthcare_fraud_detection.csv',        # Cópia no repositório: executando de dentro de PoC/
    'PoC/base_de_dados/healthcare_fraud_detection.csv',    # Executando da raiz do repositório
]
COLUNAS_CATEGORICAS_HC = ['Patient_Gender', 'Diagnosis_Code', 'Procedure_Code', 'Insurance_Type',
                          'Provider_Specialty', 'Patient_State', 'Claim_Status', 'Visit_Type']


def preparar_healthcare(df):
    df = df.dropna(subset=['Is_Fraud'])  # sem amostragem, como a base Credit Card neste teste
    df = df.copy()
    df['Is_Fraud'] = df['Is_Fraud'].astype(int)
    # Identificadores não generalizam (como nameOrig/nameDest); a data é um carimbo de tempo (como Time)
    df = df.drop(columns=['Provider_ID', 'Claim_ID', 'Claim_Submission_Date'], errors='ignore')
    # Log dos valores (como amount_log) e proporção aprovada pela operadora (análoga ao errorBalance da PaySim)
    df['approved_ratio'] = df['Approved_Amount'] / df['Claim_Amount'].replace(0, np.nan)
    df['claim_amount_log'] = np.log1p(df['Claim_Amount'].clip(lower=0))
    df['approved_amount_log'] = np.log1p(df['Approved_Amount'].clip(lower=0))
    df = df.drop(columns=['Claim_Amount', 'Approved_Amount'])
    # Categóricas -> One-Hot (Procedure_Code é código, não quantidade); ausente vira 'Desconhecido'
    for col in COLUNAS_CATEGORICAS_HC:
        df[col] = df[col].astype('object').where(df[col].notna(), 'Desconhecido').astype(str)
    df = pd.get_dummies(df, columns=COLUNAS_CATEGORICAS_HC, drop_first=True, dtype=float)
    # Numéricas ausentes (Prior_Visits_12m) -> mediana
    for col in df.select_dtypes(include=np.number).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    return df, 'Is_Fraud'


PREPARADORES['healthcare'] = preparar_healthcare
DATASETS_HEALTHCARE = ['healthcare']
print("Base healthcare registrada:", CAMINHOS['healthcare'][0])


In [ ]:
# =====================================================================
# --- 5b. EXPERIMENTO NA BASE HEALTHCARE (mesma configuração da seção 5) ---
# =====================================================================

MECANISMOS = [AgregadorClaro, AgregadorSMPC, AgregadorCKKS]
METRICAS = ['ROC-AUC', 'PR-AUC', 'Precisão', 'Recall', 'F1', 'Acurácia']
CORES = {'FedAvg em claro': '#1f77b4', 'SMPC': '#ff7f0e', 'CKKS': '#2ca02c'}
resultados_gerais = globals().get('resultados_gerais', {})  # preserva os resultados das outras bases

for nome_base in DATASETS_HEALTHCARE:
    print("\n" + "=" * 78)
    print(f" BASE: {nome_base.upper()}")
    print("=" * 78)
    dados = carregar_dados(nome_base)
    if dados is None:
        print(f"Aviso: base '{nome_base}' não encontrada em {CAMINHOS[nome_base]}. Pulando.")
        continue

    # Pesos iniciais idênticos para os três mecanismos
    seed_everything(SEED)
    estado_inicial = copy.deepcopy(TabularAutoencoder(dados['input_dim']).to(device).state_dict())
    n_params = sum(t.numel() for t in estado_inicial.values())
    print(f"Autoencoder: {dados['input_dim']} features de entrada, {n_params:,} parâmetros treináveis.")

    execucoes = {}
    for Mecanismo in MECANISMOS:
        agregador = Mecanismo()
        print(f"\n--- {agregador.nome} (setup: {agregador.tempo_setup:.2f}s) ---")
        modelo, historico = executar_fl(agregador, dados, estado_inicial)
        execucoes[agregador.nome] = dict(modelo=modelo, historico=historico, vetor=modelo_para_vetor(modelo),
                                         avaliacao=avaliar(modelo, dados), tempo_setup=agregador.tempo_setup)

    # ----- Tabela comparativa -----
    claro = execucoes['FedAvg em claro']
    linhas = []
    for nome_mec, ex in execucoes.items():
        av, hist = ex['avaliacao'], ex['historico']
        overhead_ms = 1000 * (hist['t_protecao'] + hist['t_agregacao'] + hist['t_revelacao']).mean()
        linha = {'Mecanismo': nome_mec, **{m: av[m] for m in METRICAS},
                 'Erro máx. agregação': hist['erro_max'].max(),
                 'Δ máx. pesos finais vs claro': (ex['vetor'] - claro['vetor']).abs().max().item(),
                 'Concordância pred. vs claro (%)': 100 * np.mean(av['y_pred'] == claro['avaliacao']['y_pred']),
                 'Correlação scores vs claro': np.corrcoef(av['scores'], claro['avaliacao']['scores'])[0, 1],
                 'Overhead agregação (ms/rodada)': overhead_ms,
                 'Upload por cliente/rodada (KB)': hist['bytes_cliente'].iloc[0] / 1024,
                 'Setup (s)': ex['tempo_setup'],
                 'Treino local (s/rodada)': hist['t_treino'].mean()}
        linhas.append(linha)
    tabela = pd.DataFrame(linhas).set_index('Mecanismo')
    resultados_gerais[nome_base] = dict(tabela=tabela, execucoes=execucoes)

    pd.set_option('display.width', 200)
    print("\n>> MÉTRICAS DE QUALIDADE NO TESTE (limiar escolhido na validação):")
    print(tabela[METRICAS].map(lambda x: f"{x:.4f}").to_string())
    print("\n>> FIDELIDADE E CUSTO DA AGREGAÇÃO SEGURA:")
    print(tabela.drop(columns=METRICAS).to_string(float_format=lambda x: f"{x:.4g}"))

    print("\n>> COMPARAÇÕES PAREADAS (Δ = mecanismo seguro − claro):")
    for seguro in ['SMPC', 'CKKS']:
        t = tabela.loc[seguro]
        deltas = " | ".join(f"Δ{m}: {t[m] - tabela.loc['FedAvg em claro', m]:+.2e}" for m in ['ROC-AUC', 'PR-AUC', 'F1'])
        fator = t['Overhead agregação (ms/rodada)'] / max(tabela.loc['FedAvg em claro', 'Overhead agregação (ms/rodada)'], 1e-9)
        banda = t['Upload por cliente/rodada (KB)'] / tabela.loc['FedAvg em claro', 'Upload por cliente/rodada (KB)']
        print(f"  Claro × {seguro}: {deltas} | concordância {t['Concordância pred. vs claro (%)']:.3f}% "
              f"| overhead {fator:,.0f}x | banda {banda:.1f}x")

    # ----- Visualizações -----
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    fig.suptitle(f"Teste 5: FedAvg em claro × SMPC × CKKS ({nome_base})", fontsize=14, fontweight='bold')

    # 1. Métricas de qualidade
    ax = axes[0, 0]
    tabela[METRICAS].T.plot.bar(ax=ax, color=[CORES[m] for m in tabela.index], rot=0)
    ax.set_title('Métricas no teste')
    ax.set_ylim(0, 1.2)
    ax.legend(loc='upper center', ncol=3)

    # 2. Convergência
    ax = axes[0, 1]
    for nome_mec, ex in execucoes.items():
        estilo = '-' if nome_mec == 'FedAvg em claro' else '--'
        ax.plot(ex['historico']['rodada'], ex['historico']['roc_auc_val'], estilo, marker='o',
                color=CORES[nome_mec], label=nome_mec)
    ax.set_title('ROC-AUC na validação por rodada')
    ax.set_xlabel('Rodada')
    ax.legend()

    # 3. Curvas ROC no teste
    ax = axes[0, 2]
    for nome_mec, ex in execucoes.items():
        fpr, tpr, _ = roc_curve(dados['y_test'], ex['avaliacao']['scores'])
        estilo = '-' if nome_mec == 'FedAvg em claro' else '--'
        ax.plot(fpr, tpr, estilo, color=CORES[nome_mec], label=f"{nome_mec} (AUC={ex['avaliacao']['ROC-AUC']:.4f})")
    ax.plot([0, 1], [0, 1], color='gray', linestyle=':')
    ax.set_title('Curva ROC no teste')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(loc='lower right')

    # 4. Erro numérico da agregação
    ax = axes[1, 0]
    for nome_mec in ['SMPC', 'CKKS']:
        h = execucoes[nome_mec]['historico']
        ax.plot(h['rodada'], h['erro_max'], marker='o', color=CORES[nome_mec], label=f'{nome_mec} (máx.)')
        ax.plot(h['rodada'], h['erro_rmse'], marker='x', linestyle=':', color=CORES[nome_mec], label=f'{nome_mec} (RMSE)')
    ax.set_yscale('log')
    ax.set_title('Erro da agregação vs FedAvg exato')
    ax.set_xlabel('Rodada')
    ax.legend()

    # 5. Tempo de agregação por etapa
    ax = axes[1, 1]
    tempos = pd.DataFrame({nome_mec: 1000 * ex['historico'][['t_protecao', 't_agregacao', 't_revelacao']].mean()
                           for nome_mec, ex in execucoes.items()}).T
    tempos.columns = ['Proteção (por cliente)', 'Agregação', 'Revelação']
    tempos.plot.bar(stacked=True, ax=ax, rot=0, color=['#9467bd', '#8c564b', '#e377c2'])
    ax.set_yscale('symlog', linthresh=0.1)
    ax.set_title('Custo da agregação por rodada (ms)')

    # 6. Comunicação
    ax = axes[1, 2]
    kb = tabela['Upload por cliente/rodada (KB)']
    ax.bar(kb.index, kb.values, color=[CORES[m] for m in kb.index])
    for i, v in enumerate(kb.values):
        ax.text(i, v, f"{v:,.1f} KB", ha='center', va='bottom')
    ax.set_yscale('log')
    ax.set_title('Upload por cliente por rodada')

    plt.tight_layout()
    plt.show()

print("\nFim do Teste 5 (FedAvg em claro × SMPC × CKKS) na base healthcare!")

In [ ]:
# =====================================================================
# --- COMPARATIVO ENTRE AS BASES (Teste 5: FedAvg em claro × SMPC × CKKS) ---
# =====================================================================
import contextlib
import io

NOMES_BASES = {'creditcard': 'Credit Card', 'paysim': 'PaySim', 'healthcare': 'Healthcare'}
# Cor fixa por base (paleta categórica validada para daltonismo): a mesma base tem sempre a mesma cor
CORES_BASES = {'creditcard': '#2a78d6', 'healthcare': '#eb6834', 'paysim': '#1baf7a'}
bases = [b for b in NOMES_BASES if b in globals().get('resultados_gerais', {})]


def perfil_da_base(nome):
    """Tamanho e taxa de fraude do conjunto de teste (a base é recarregada só para essas estatísticas)."""
    with contextlib.redirect_stdout(io.StringIO()):
        d = carregar_dados(nome)
    return {'Transações (teste)': len(d['y_test']), 'Taxa de fraude (teste)': float(np.mean(d['y_test'])),
            'Features': int(d['input_dim'])}

ORDEM_MECANISMOS = ['FedAvg em claro', 'SMPC', 'CKKS']

if not bases:
    print("Nenhum resultado disponível: execute a seção 5 e/ou a base healthcare antes do comparativo.")
else:
    perfis = {b: perfil_da_base(b) for b in bases}
    registros = []
    for b in bases:
        for mecanismo, linha in resultados_gerais[b]['tabela'].iterrows():
            registros.append({'Base': NOMES_BASES[b], 'Mecanismo': mecanismo, **linha.to_dict()})
    longo = pd.DataFrame(registros)

    print(">> CARACTERÍSTICAS DAS BASES")
    for b in bases:
        p = perfis[b]
        print(f"  {NOMES_BASES[b]:12s} teste: {p['Transações (teste)']:,} transações | taxa de fraude {p['Taxa de fraude (teste)']:.2%} "
              f"| {p['Features']} features")

    print("\n>> MÉTRICAS NO TESTE POR MECANISMO E BASE")
    for m in ['ROC-AUC', 'PR-AUC', 'F1']:
        print(f"\n{m}:")
        print(longo.pivot(index='Mecanismo', columns='Base', values=m).reindex(ORDEM_MECANISMOS).round(4).to_string())

    print("\n>> EFEITO DA AGREGAÇÃO SEGURA (mecanismo seguro − FedAvg em claro)")
    for b in bases:
        t = resultados_gerais[b]['tabela']
        for seguro in ['SMPC', 'CKKS']:
            deltas = " | ".join(f"Δ{m} {t.loc[seguro, m] - t.loc['FedAvg em claro', m]:+.1e}" for m in ['ROC-AUC', 'PR-AUC', 'F1'])
            print(f"  {NOMES_BASES[b]:12s} Claro × {seguro}: {deltas} | concordância das predições "
                  f"{t.loc[seguro, 'Concordância pred. vs claro (%)']:.3f}%")

    print("\n>> CUSTO DA AGREGAÇÃO POR BASE (cresce com o nº de parâmetros, que depende do nº de features)")
    for coluna in ['Overhead agregação (ms/rodada)', 'Upload por cliente/rodada (KB)', 'Erro máx. agregação']:
        print(f"\n{coluna}:")
        print(longo.pivot(index='Mecanismo', columns='Base', values=coluna).reindex(ORDEM_MECANISMOS)
              .to_string(float_format=lambda v: f"{v:.4g}"))
    print("\nObservação: as bases têm taxas de fraude muito diferentes. O PR-AUC de um classificador aleatório é igual "
          "à taxa de fraude, por isso 'PR-AUC / taxa de fraude' (ganho sobre o acaso) é a comparação justa do PR-AUC.")

    fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))
    largura = 0.8 / len(bases)
    x = np.arange(len(ORDEM_MECANISMOS))
    for eixo, (coluna, titulo) in zip(axes, [('ROC-AUC', 'ROC-AUC no teste'), ('PR-AUC', 'PR-AUC no teste'),
                                             ('Overhead agregação (ms/rodada)', 'Custo da agregação (ms/rodada, escala log)')]):
        for i, b in enumerate(bases):
            valores = resultados_gerais[b]['tabela'].reindex(ORDEM_MECANISMOS)[coluna].values
            eixo.bar(x + (i - (len(bases) - 1) / 2) * largura, valores, largura, color=CORES_BASES[b], label=NOMES_BASES[b], edgecolor='white', linewidth=1.5)
        eixo.set_xticks(x)
        eixo.set_xticklabels(ORDEM_MECANISMOS)
        eixo.set_title(titulo, fontweight='bold')
        eixo.legend()
    axes[0].set_ylim(0, 1.05)
    axes[1].set_ylim(0, 1.05)
    axes[2].set_yscale('log')
    plt.suptitle('Teste 5 — FedAvg em claro × SMPC × CKKS: comparativo entre as bases', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
